# Manos al código: transformers desde R

**Extensión opcional · 60–90 minutos**

Este cuaderno recorre los mismos objetos que la versión Python desde R. Usamos `reticulate` porque el ecosistema de Hugging Face y PyTorch expone así, sin una capa conceptual extra, los estados ocultos y las atenciones. No hace falta saber Python para ejecutar y modificar los ejemplos.

## 0. Preparación
Necesitás R, Jupyter con kernel de R y un entorno Python visible para `reticulate`. La instalación de los paquetes Python se hace una sola vez. Si ya tenés un entorno configurado, podés omitir `py_install()`.

In [ ]:
needed <- setdiff(c("reticulate", "ggplot2"), rownames(installed.packages()))
if (length(needed)) install.packages(needed, repos = "https://cloud.r-project.org")
library(reticulate)
library(ggplot2)

# Ejecutá esta línea solo si el entorno todavía no tiene las dependencias:
# py_install(c("transformers>=4.46,<5", "torch"), pip = TRUE)
py_config()

In [ ]:
transformers <- import("transformers", convert = FALSE)
torch <- import("torch", convert = FALSE)
np <- import("numpy", convert = FALSE)
torch$manual_seed(7L)

## 1. Generar texto con una `pipeline`
La `pipeline` reúne tokenizador, modelo y posprocesamiento. Una temperatura baja concentra el muestreo; para elegir siempre el token más probable se usa `do_sample = FALSE`.

In [ ]:
generator <- transformers$pipeline(
  "text-generation",
  model = "HuggingFaceTB/SmolLM2-135M-Instruct",
  device = -1L
)

prompt <- "Explicá qué es un token en dos oraciones y con un ejemplo en español."
result <- generator(
  prompt, max_new_tokens = 70L, do_sample = TRUE, temperature = 0.7,
  return_full_text = FALSE, pad_token_id = generator$tokenizer$eos_token_id
)
py_to_r(result)[[1]]$generated_text

**Probá:** volvé a ejecutar, compará `temperature = 0.2` con `1.3` y finalmente usá `do_sample = FALSE`.

## 2. Tokenización y una pasada por el encoder
Cargamos un DistilBERT multilingüe y pedimos estados ocultos y atenciones. `torch$no_grad()` evita calcular gradientes durante esta inferencia.

In [ ]:
encoder_name <- "distilbert-base-multilingual-cased"
tokenizer <- transformers$AutoTokenizer$from_pretrained(encoder_name)
model <- transformers$AutoModel$from_pretrained(
  encoder_name, attn_implementation = "eager"
)
model$eval()
torch$set_grad_enabled(FALSE)

text <- "La tokenización puede partir palabras desconocidas."
inputs <- tokenizer(text, return_tensors = "pt")
ids <- py_get_item(inputs$input_ids, 0L)$tolist()
tokens <- py_to_r(tokenizer$convert_ids_to_tokens(ids))
data.frame(token = tokens, id = py_to_r(ids))

outputs <- model(
  input_ids = inputs$input_ids,
  attention_mask = inputs$attention_mask,
  output_hidden_states = TRUE,
  output_attentions = TRUE
)

cat("Forma de la activación final:", paste(py_to_r(outputs$last_hidden_state$shape), collapse = " × "), "\n")
cat("Estados (embedding + capas):", py_len(outputs$hidden_states), "\n")
cat("Matrices de atención:", py_len(outputs$attentions), "\n")

## 3. Embedding de entrada vs. activación contextual
El embedding inicial surge de buscar el ID en una matriz aprendida. La activación final ya fue transformada usando el resto de la oración.

In [ ]:
input_embeddings <- model$get_input_embeddings()(inputs$input_ids)
initial <- py_to_r(input_embeddings$detach()$cpu()$numpy())
contextual <- py_to_r(outputs$last_hidden_state$detach()$cpu()$numpy())

position <- 4L  # los arrays de R comienzan en 1
cat("Token elegido:", tokens[position], "\n")
round(initial[1, position, 1:8], 3)
round(contextual[1, position, 1:8], 3)
cat("Distancia entre ambos:", sqrt(sum((initial[1, position, ] - contextual[1, position, ])^2)), "\n")

## 4. Un embedding para cada oración
Hacemos *mean pooling*: promediamos las activaciones de los tokens reales e ignoramos el *padding*. Es una demostración sencilla, no una receta universal para búsqueda semántica.

In [ ]:
sentences <- c(
  "El gato duerme sobre el sillón.",
  "Un felino descansa en el sofá.",
  "La inflación anual volvió a bajar."
)
batch <- tokenizer(r_to_py(sentences), return_tensors = "pt", padding = TRUE, truncation = TRUE)
batch_outputs <- model(input_ids = batch$input_ids, attention_mask = batch$attention_mask)
hidden <- py_to_r(batch_outputs$last_hidden_state$detach()$cpu()$numpy())
mask <- py_to_r(batch$attention_mask$detach()$cpu()$numpy())

sentence_embeddings <- t(vapply(seq_along(sentences), function(i) {
  active <- which(mask[i, ] == 1)
  colMeans(hidden[i, active, , drop = FALSE][1, , ])
}, numeric(dim(hidden)[3])))

cosine <- function(a, b) sum(a * b) / sqrt(sum(a^2) * sum(b^2))
similarities <- outer(seq_along(sentences), seq_along(sentences), Vectorize(function(i, j) {
  cosine(sentence_embeddings[i, ], sentence_embeddings[j, ])
}))
round(similarities, 3)

In [ ]:
labels <- c("gato", "felino", "inflación")
heat <- expand.grid(x = labels, y = labels)
heat$similarity <- as.vector(similarities)
ggplot(heat, aes(x, y, fill = similarity)) +
  geom_tile() + geom_text(aes(label = round(similarity, 2))) +
  scale_fill_gradient(low = "#f7f6ef", high = "#176f62", limits = c(0, 1)) +
  coord_equal() + labs(title = "Similitud coseno entre embeddings", x = NULL, y = NULL) +
  theme_minimal()

## 5. Cómo cambian las activaciones capa a capa
Cada estado tiene forma `(lote, tokens, dimensiones)`. Resumimos cada capa mediante la norma L2 media de sus vectores.

In [ ]:
n_states <- py_len(outputs$hidden_states)
mean_norm <- vapply(0:(n_states - 1L), function(i) {
  state <- py_get_item(outputs$hidden_states, as.integer(i))
  array <- py_to_r(state$detach()$cpu()$numpy())[1, , ]
  mean(sqrt(rowSums(array^2)))
}, numeric(1))

activation_df <- data.frame(
  layer = factor(0:(n_states - 1L), labels = c("entrada", paste("capa", 1:(n_states - 1L)))),
  mean_norm = mean_norm
)
ggplot(activation_df, aes(layer, mean_norm, group = 1)) +
  geom_line(color = "#176f62") + geom_point(color = "#f08a70", size = 3) +
  labs(title = "Magnitud de las activaciones a través del encoder", x = NULL, y = "Norma L2 media") +
  theme_minimal()

## 6. Una cabeza de atención
Extraemos la primera cabeza de la última capa. Las filas son tokens que consultan; las columnas, tokens de los que toman información. **Atención no equivale automáticamente a explicación**.

In [ ]:
last_layer <- py_get_item(outputs$attentions, as.integer(py_len(outputs$attentions) - 1L))
attention_array <- py_to_r(last_layer$detach()$cpu()$numpy())
head_1 <- attention_array[1, 1, , ]

attention_df <- expand.grid(key = tokens, query = tokens)
attention_df$weight <- as.vector(t(head_1))
ggplot(attention_df, aes(key, query, fill = weight)) +
  geom_tile() + scale_fill_gradient(low = "#f7f6ef", high = "#132d35") +
  coord_equal() + labs(title = "Atención · última capa, primera cabeza", x = "key", y = "query") +
  theme_minimal() + theme(axis.text.x = element_text(angle = 45, hjust = 1))

## Para cerrar
Ahora podés distinguir: IDs de tokens, embeddings iniciales, activaciones contextuales y pesos de atención.

**Desafío opcional:** cambiá el texto y la cabeza de atención. Describí el patrón que observás, separando cuidadosamente *visualización* de *explicación causal*.